# Do the models notice an incorrect proof step?

## 1. Load the proof pairs

Evaluate the five theorem-and-proof models from notebooks 01–05. The 16 originals were manually retained from test proofs all five models classified as Valid. Each has one main incorrect version. Three longer proofs also have one-, two-, and three-symbol edits and a mathematically valid rewrite. The edited labels and explanations were prepared by AI. Only the theorem and proof are model inputs.

In [1]:
import json
import os
import sys
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed

sys.path.insert(0, str(Path("..").resolve()))
from src.proof_experiments import LABELS, load_split

results = Path("../Results")
groups = json.loads((results / "proof_edit_challenge.json").read_text(encoding="utf-8"))
records = []
for group in groups:
    row_id = group["clean_row_index"]
    original = {"variant_id": f"{row_id}-original", "proof_text": group["original_proof_text"],
                "validity": "Valid", "edit_reason": ""}
    for version in [original, *group["variants"]]:
        records.append({**version, "clean_row_index": row_id, "theorem_id": group["theorem_id"],
                        "title": group["title"], "theorem_text": group["theorem_text"]})

proofs = pd.DataFrame(records)
assert proofs["variant_id"].is_unique
predictions = proofs[["clean_row_index", "theorem_id", "variant_id", "validity"]].copy()
train = load_split("train")
validation = load_split("validation")
test = load_split("test")
assert set(proofs["theorem_id"]).isdisjoint(train["theorem_id"])
assert set(proofs["clean_row_index"]).issubset(test["clean_row_index"])
print(f"{len(groups)} original proofs and {len(proofs) - len(groups)} edited versions.")

C:\Users\zhu25\AppData\Local\DSA495ProofVerification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


16 original proofs and 26 edited versions.


## 2. Predict with the TF-IDF baseline

Refit the configuration selected in notebook 01 on the original training split. Check the original test confusion matrix before scoring the edits.

In [2]:
def model_input(rows):
    return rows["theorem_text"] + "\n" + rows["proof_text"]

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=False, min_df=2),
    LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000),
)
baseline.fit(model_input(train), train["validity"])
test_pred = baseline.predict(model_input(test))
assert confusion_matrix(test["validity"], test_pred, labels=LABELS).tolist() == [[54, 21], [9, 5]]
predictions["TF-IDF"] = baseline.predict(model_input(proofs))
# Keep raw scores for the model visualizations alongside the native model decisions.
score_rows = []
def record_scores(model_name, dataset, rows, decisions, invalid_scores):
    score_rows.append(pd.DataFrame({
        "model": model_name, "dataset": dataset, "clean_row_index": rows["clean_row_index"].to_numpy(),
        "theorem_id": rows["theorem_id"].to_numpy(),
        "variant_id": rows["variant_id"].to_numpy() if "variant_id" in rows else "test",
        "actual": rows["validity"].to_numpy(), "predicted": decisions, "invalid_score": invalid_scores,
    }))

invalid_column = list(baseline.classes_).index("Invalid")
for dataset, rows, decisions in (("test", test, test_pred), ("challenge", proofs, predictions["TF-IDF"])):
    record_scores("TF-IDF", dataset, rows, decisions, baseline.predict_proba(model_input(rows))[:, invalid_column])
print("TF-IDF predictions complete.")

TF-IDF predictions complete.


## 3. Reproduce the selected BERT models

The earlier experiments saved their settings and predictions. Refit only the selected learning rate and epoch count on the original training proofs. Use the same token limits, optimizer, batch order and seed as notebooks 02–05. Verify every original test prediction before predicting the edited proofs.

In [3]:
device = torch.device("cuda")
torch.use_deterministic_algorithms(True)

def keep_ends(ids, limit):
    if len(ids) <= limit:
        return ids
    first = (limit + 1) // 2
    return ids[:first] + ids[-(limit - first):]

def encode(rows, tokenizer, settings):
    theorems = tokenizer(rows["theorem_text"].tolist(), add_special_tokens=False, verbose=False)["input_ids"]
    proofs_ids = tokenizer(rows["proof_text"].tolist(), add_special_tokens=False, verbose=False)["input_ids"]
    budget = settings["max_length"] - settings["reason_limit"] - 4
    encoded = []
    for theorem, proof in zip(theorems, proofs_ids):
        theorem = keep_ends(theorem, settings["theorem_limit"])
        proof = keep_ends(proof, budget - len(theorem))
        sep = tokenizer.sep_token_id
        encoded.append([tokenizer.cls_token_id] + theorem + [sep] + proof + [sep])
    return encoded

@torch.inference_mode()
def classify(model, tokenizer, encoded, batch_size):
    model.eval()
    predicted = []
    invalid_scores = []
    with torch.autocast("cuda", dtype=torch.bfloat16):
        for start in range(0, len(encoded), batch_size):
            batch = tokenizer.pad({"input_ids": encoded[start:start + batch_size]}, padding=True,
                                  pad_to_multiple_of=8, return_tensors="pt").to(device)
            logits = model(**batch).logits.float()
            predicted.extend(LABELS[i] for i in logits.argmax(dim=1).tolist())
            invalid_scores.extend(logits.softmax(dim=1)[:, LABELS.index("Invalid")].tolist())
    return predicted, invalid_scores

In [4]:
def fit_selected(name, tokenizer, settings):
    best = settings["runs"]["without_review_reason"]["selected"]
    batch_size = settings["batch_size"]
    set_seed(settings["seed"])
    options = {"reference_compile": False, "attn_implementation": "sdpa"} if name == "ModernBERT-base" else {}
    model = AutoModelForSequenceClassification.from_pretrained(
        settings["model"], revision=settings["revision"], id2label=dict(enumerate(LABELS)),
        label2id={label: i for i, label in enumerate(LABELS)}, **options,
    ).to(device)
    ids = encode(train, tokenizer, settings)
    targets = [LABELS.index(label) for label in train["validity"]]
    loader = DataLoader(range(len(ids)), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(settings["seed"]))
    weights = [settings["class_weights"][label] for label in LABELS]
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, device=device, dtype=torch.float32))
    optimizer = torch.optim.AdamW(model.parameters(), lr=best["learning_rate"], weight_decay=settings["weight_decay"])

    for epoch in range(1, best["epoch"] + 1):
        model.train()
        for indices in loader:
            batch = tokenizer.pad({"input_ids": [ids[i] for i in indices]}, padding=True,
                                  pad_to_multiple_of=8, return_tensors="pt").to(device)
            labels = torch.tensor([targets[i] for i in indices], device=device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(**batch).logits
            loss_fn(logits.float(), labels).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), settings["gradient_clip_norm"])
            optimizer.step()
        print(f"{name}: completed epoch {epoch} / {best['epoch']}")
    return model

In [5]:
bert_names = ["DistilBERT", "ModernBERT-base", "MathBERT", "DeBERTa-v3-base"]
for name in bert_names:
    folder = results / name
    settings = json.loads((folder / "experiment.json").read_text(encoding="utf-8"))
    tokenizer = AutoTokenizer.from_pretrained(settings["model"], revision=settings["revision"],
                                               use_fast=(name != "DeBERTa-v3-base"))
    model = fit_selected(name, tokenizer, settings)
    saved = pd.read_csv(folder / "predictions.csv").query("training_input == 'without_review_reason'")
    for split, data in (("validation", validation), ("test", test)):
        expected = saved[saved["split"] == split].set_index("clean_row_index").loc[data["clean_row_index"], "predicted"]
        actual, invalid_scores = classify(model, tokenizer, encode(data, tokenizer, settings), settings["batch_size"])
        assert actual == expected.tolist(), f"{name}: original {split} predictions differ"

        if split == "test":
            record_scores(name, split, data, actual, invalid_scores)

    encoded = encode(proofs, tokenizer, settings)
    predictions[name], invalid_scores = classify(model, tokenizer, encoded, settings["batch_size"])
    record_scores(name, "challenge", proofs, predictions[name], invalid_scores)
    token_ids = dict(zip(proofs["variant_id"], encoded))
    predictions[name + " input changed"] = [
        ids != token_ids[f"{row_id}-original"] for ids, row_id in zip(encoded, proofs["clean_row_index"])
    ]
    print(f"{name}: verified original predictions and scored {len(encoded)} versions.")
    del model
    torch.cuda.empty_cache()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


DistilBERT: completed epoch 1 / 2
DistilBERT: completed epoch 2 / 2
DistilBERT: verified original predictions and scored 42 versions.


W1122 22:24:41.755000 4492 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


ModernBERT-base: completed epoch 1 / 1
ModernBERT-base: verified original predictions and scored 42 versions.


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at tbs17/MathBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


MathBERT: completed epoch 1 / 1
MathBERT: verified original predictions and scored 42 versions.


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DeBERTa-v3-base: completed epoch 1 / 2
DeBERTa-v3-base: completed epoch 2 / 2
DeBERTa-v3-base: verified original predictions and scored 42 versions.


## 4. Compare original and edited proofs

Count a pair as correct only when the model accepts the original and flags its main edit. A valid rewrite should still be accepted. The denominator for the main comparison is the 16 selected proof versions.

In [6]:
models = ["TF-IDF", *bert_names]
by_id = predictions.set_index("variant_id")
source_ids = [g["clean_row_index"] for g in groups]
original = by_id.loc[[f"{i}-original" for i in source_ids], models]
edited = by_id.loc[[f"{i}-primary" for i in source_ids], models]
controls = by_id.loc[by_id.index.str.endswith("-valid-control"), models]
assert original.eq("Valid").all().all()
assert edited.index.size == len(source_ids) == 16 and len(controls) == 3
pair_correct = (original.to_numpy() == "Valid") & (edited.to_numpy() == "Invalid")
summary = pd.DataFrame({"Originals accepted / 16": original.eq("Valid").sum(),
                        "Both versions correct / 16": pair_correct.sum(axis=0),
                        "Valid rewrites accepted / 3": controls.eq("Valid").sum()})
display(summary)
print(f"Main edits missed by all five models: {edited.eq('Valid').all(axis=1).sum()} / {len(edited)}")
predictions.to_csv(results / "proof_edit_predictions.csv", index=False)
scores = pd.concat(score_rows, ignore_index=True)
assert scores["invalid_score"].between(0, 1).all()
assert not scores.duplicated(["model", "dataset", "clean_row_index", "variant_id"]).any()
scores.to_csv(results / "proof_scores.csv", index=False)

,Originals accepted / 16,Both versions correct / 16,Valid rewrites accepted / 3
TF-IDF,16,2,3
DistilBERT,16,3,3
ModernBERT-base,16,6,3
MathBERT,16,4,3
DeBERTa-v3-base,16,8,3


Main edits missed by all five models: 8 / 16


## 5. Compare one, two and three symbol edits

Each row below changes the indicated number of symbols in the same original proof. For rows 84 and 390 the first symbol edit is also their main edit. The original of row 531 has a separate main edit about assuming a minimum exists.

In [7]:
symbol_rows = proofs.dropna(subset=["changed_symbol_count"]).copy()
symbol_rows["Symbols changed"] = symbol_rows["changed_symbol_count"].astype(int)
symbol_rows = symbol_rows.join(by_id[models], on="variant_id")
display(symbol_rows.set_index(["clean_row_index", "Symbols changed"])[models])

TF-IDF DistilBERT ModernBERT-base MathBERT  \
clean_row_index Symbols changed                                              
84              1                Valid      Valid           Valid    Valid   
                2                Valid      Valid           Valid    Valid   
                3                Valid      Valid           Valid    Valid   
390             1                Valid      Valid           Valid    Valid   
                2                Valid      Valid           Valid    Valid   
                3                Valid      Valid           Valid    Valid   
531             1                Valid      Valid           Valid    Valid   
                2                Valid      Valid           Valid    Valid   
                3                Valid      Valid           Valid    Valid   

                                DeBERTa-v3-base  
clean_row_index Symbols changed                  
84              1                         Valid  
                2                         Valid  
                3                         Valid  
390             1                         Valid  
                2                         Valid  
                3                         Valid  
531             1                         Valid  
                2                         Valid  
                3                         Valid

## 6. Read an individual proof

Run this cell on its own to read the saved predictions. Change the variant ID to inspect another edit, for example `531-primary`, `84-symbol-2` or `390-valid-control`.

In [8]:
import json
from pathlib import Path
import pandas as pd

variant_id = "390-primary"
results_dir = Path("../Results")
groups = json.loads((results_dir / "proof_edit_challenge.json").read_text(encoding="utf-8"))
saved_predictions = pd.read_csv(results_dir / "proof_edit_predictions.csv").set_index("variant_id")
group = next(group for group in groups if group["clean_row_index"] == int(variant_id.split("-")[0]))
edit = next(version for version in group["variants"] if version["variant_id"] == variant_id)

print(group["title"], "\n", group["theorem_text"])
print("\nOriginal proof\n", group["original_proof_text"])
print("\nEdited proof\n", edit["proof_text"])
print("\nExpected label:", edit["validity"])
print(edit["edit_reason"])
models = ["TF-IDF", "DistilBERT", "ModernBERT-base", "MathBERT", "DeBERTa-v3-base"]
original_id = f"{group['clean_row_index']}-original"
display(saved_predictions.loc[[original_id, variant_id], ["validity", *models]])

Reduced Residue System is Subset of Set of All Residue Classes 
 Let $\Z_m$ be the set of set of residue classes modulo $m$.
Let $\Z'_m$ be the reduced residue system modulo $m$.
Then:
$\forall m \in \Z_{> 1}: \O \subset \Z'_m \subset \Z_m$

Original proof
 By definition of reduced residue system modulo $m$:
$\Z'_m = \set {x \in \Z_m: x \perp m}$
From Subset of Set with Propositional Function:
$\Z'_m \subseteq \Z_m$
We have that:
$\gcd \set {m, 0} = m$
Thus it follows that:
$m > 1 \implies \gcd \set {m, 0} \ne 1$
So:
$\eqclass 0 m \notin \Z'_m$
However:
$\eqclass 0 m \in \Z_m$
so:
$\Z'_m \ne \Z_m$
Thus:
$\Z'_m \subset \Z_m$
Then:
$\eqclass 1 m = 1 \implies 1 \perp m$
So:
$\forall m \in \Z: \eqclass 1 m \in \Z'_m$
Thus:
$\Z'_m \ne \O$
and therefore:
$\O \subset \Z'_m$

Edited proof
 By definition of reduced residue system modulo $m$:
$\Z'_m = \set {x \in \Z_m: x \perp m}$
From Subset of Set with Propositional Function:
$\Z'_m \subseteq \Z_m$
We have that:
$\gcd \set {m, 0} = m$
Thus it 

,validity,TF-IDF,DistilBERT,ModernBERT-base,MathBERT,DeBERTa-v3-base
variant_id,,,,,,
390-original,Valid,Valid,Valid,Valid,Valid,Valid
390-primary,Invalid,Valid,Valid,Valid,Valid,Valid
